In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import simpson
from scipy.optimize import brentq

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from scipy.interpolate import griddata
import glob
import os

# ==============================================================================
# 1. VISUAL CONFIGURATION (STIX Fonts)
# ==============================================================================
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 16
mpl.rcParams['figure.dpi'] = 100

# Load the time array from saved data
data = np.load('../Results/NM_codes/Wave_Neumann_GaussianSin/2026_01_11_1252/wave_spherical.npz')
t_1d = data['t']

print(f"Loaded time array: {t_1d.shape}")

# -----------------------------------------------------------------------------
# Recompute eigenvalues and coefficients
# -----------------------------------------------------------------------------

def find_eigenvalues(L, num_modes):
    """Find eigenvalues for the spherical wave problem."""
    f = lambda k: np.tan(L*k) - L*k
    k_roots = []
    
    for n in range(1, num_modes + 1):
        z_min = n * np.pi + 1e-6
        z_max = n * np.pi + np.pi/2 - 1e-6
        k_min = z_min / L
        k_max = z_max / L
        
        try:
            root = brentq(f, k_min, k_max)
            k_roots.append(root)
        except:
            pass
    
    return np.array(k_roots)

def compute_coefficients(k_vals, L):
    """Compute Fourier coefficients."""
    r_int = np.linspace(0, L, 10000)
    
    # Initial condition: v(r,0) = exp(-10(r-5)^2) * sin(pi*r/L)
    v0_vals = np.exp(-10.0 * (r_int - 5.0)**2) * np.sin(np.pi * r_int / L)
    
    A_vals = []
    
    for k in k_vals:
        phi_vals = np.sin(k * r_int)
        integrand = v0_vals * phi_vals
        num = simpson(integrand, x=r_int)
        den = (L / 2.0) - (np.sin(2.0 * k * L) / (4.0 * k))
        A_vals.append(num / den)
    
    return np.array(A_vals)

# Compute eigenvalues and coefficients
L = 10.0
num_modes = 200
k_vals = find_eigenvalues(L, num_modes)
A_vals = compute_coefficients(k_vals, L)

print(f"Computed {len(k_vals)} eigenvalues")

# -----------------------------------------------------------------------------
# Direct 2D spectral evaluation
# -----------------------------------------------------------------------------

def compute_solution_2d_spectral(k_vals, A_vals, t, N_2d=150):
    """
    Compute u(x,y,0,t) directly using spectral series.
    """
    r_max = L
    
    # Create 2D grid
    x = np.linspace(-r_max, r_max, N_2d)
    y = np.linspace(-r_max, r_max, N_2d)
    X, Y = np.meshgrid(x, y)
    
    # Compute radial distance
    R_2d = np.sqrt(X**2 + Y**2)
    
    # Initialize solution
    U_2d = np.zeros_like(R_2d)
    
    # Vectorized evaluation for speed
    for n in range(len(k_vals)):
        k = k_vals[n]
        A = A_vals[n]
        
        # Avoid division by zero at origin
        mask = R_2d > 1e-9
        U_2d[mask] += A * np.cos(k * t) * np.sin(k * R_2d[mask]) / R_2d[mask]
        
        # At origin: lim_{r->0} sin(kr)/r = k
        U_2d[~mask] += A * np.cos(k * t) * k
    
    return X, Y, U_2d

# -----------------------------------------------------------------------------
# Precompute global z-limits
# -----------------------------------------------------------------------------

def compute_global_zlims(k_vals, A_vals, t_1d, N_2d=150):
    """Compute global min/max across all time steps."""
    T_final = len(t_1d) - 1
    time_indices = [0, T_final//3, 2*T_final//3, T_final]
    
    z_min_global = np.inf
    z_max_global = -np.inf
    
    print("Computing global z-limits...")
    for time_idx in time_indices:
        t_val = t_1d[time_idx]
        _, _, U_2d = compute_solution_2d_spectral(k_vals, A_vals, t_val, N_2d)
        z_min_global = min(z_min_global, U_2d.min())
        z_max_global = max(z_max_global, U_2d.max())
    
    print(f"Global z-limits: [{z_min_global:.4f}, {z_max_global:.4f}]")
    return z_min_global, z_max_global

# -----------------------------------------------------------------------------
# Visualization with fixed z-axis
# -----------------------------------------------------------------------------

def plot_3d_evolution_row(k_vals, A_vals, t_1d, N_2d=150):
    """Plot 4 3D surfaces with FIXED z-axis limits."""
    
    # Select time indices
    T_final = len(t_1d) - 1
    time_indices = [0, T_final//3, 2*T_final//3, T_final]
    
    # Compute global z-limits FIRST
    z_min_global, z_max_global = compute_global_zlims(k_vals, A_vals, t_1d, N_2d)
    
    fig = plt.figure(figsize=(20, 5))
    
    for idx, time_idx in enumerate(time_indices, 1):
        t_val = t_1d[time_idx]
        
        print(f"Rendering t={t_val:.2f}...")
        
        # Compute solution directly using spectral series
        X, Y, U_2d = compute_solution_2d_spectral(k_vals, A_vals, t_val, N_2d)
        
        ax = fig.add_subplot(1, 4, idx, projection='3d')
        
        # Use plot_surface with full resolution
        surf = ax.plot_surface(X, Y, U_2d, 
                               cmap='viridis',
                               edgecolor='none',
                               linewidth=0,
                               antialiased=True,
                               shade=True,
                               rcount=N_2d,
                               ccount=N_2d,
                               vmin=z_min_global,  # Fixed color scale
                               vmax=z_max_global,  # Fixed color scale
                               alpha=1.0)
        
        ax.set_title(f't = {t_val:.2f}', fontsize=12, pad=10)
        
        # Set FIXED limits for all axes
        ax.set_xlim([-L, L])
        ax.set_ylim([-L, L])
        ax.set_zlim([z_min_global, z_max_global])  # FIXED z-axis
        
        ax.view_init(elev=30, azim=45)
        ax.grid(False)
        ax.set_axis_off()
        
        # Individual colorbar
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.1)
    
    plt.tight_layout()
    plt.savefig('spherical_wave_3d_evolution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nVisualization saved: spherical_wave_3d_evolution.png")

# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    print("="*60)
    print("3D Spherical Wave Evolution (Fixed Z-Axis)")
    print("="*60)
    
    import time
    start = time.time()
    plot_3d_evolution_row(k_vals, A_vals, t_1d, N_2d=150)
    print(f"Total time: {time.time() - start:.2f} seconds")
    
    print("="*60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import simpson
from scipy.optimize import brentq

import matplotlib as mpl

# ==============================================================================
# 1. VISUAL CONFIGURATION (STIX Fonts)
# ==============================================================================
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 16
mpl.rcParams['figure.dpi'] = 100

# Load the time array from saved data
data = np.load('../Results/NM_codes/Wave_Neumann_GaussianSin/2026_01_11_1252/wave_spherical.npz')
t_1d = data['t']

print(f"Loaded time array: {t_1d.shape}")

# -----------------------------------------------------------------------------
# Recompute eigenvalues and coefficients
# -----------------------------------------------------------------------------

def find_eigenvalues(L, num_modes):
    """Find eigenvalues for the spherical wave problem."""
    f = lambda k: np.tan(L*k) - L*k
    k_roots = []
    
    for n in range(1, num_modes + 1):
        z_min = n * np.pi + 1e-6
        z_max = n * np.pi + np.pi/2 - 1e-6
        k_min = z_min / L
        k_max = z_max / L
        
        try:
            root = brentq(f, k_min, k_max)
            k_roots.append(root)
        except:
            pass
    
    return np.array(k_roots)

def compute_coefficients(k_vals, L):
    """Compute Fourier coefficients."""
    r_int = np.linspace(0, L, 10000)
    
    # Initial condition: v(r,0) = exp(-10(r-5)^2) * sin(pi*r/L)
    v0_vals = np.exp(-10.0 * (r_int - 5.0)**2) * np.sin(np.pi * r_int / L)
    
    A_vals = []
    
    for k in k_vals:
        phi_vals = np.sin(k * r_int)
        integrand = v0_vals * phi_vals
        num = simpson(integrand, x=r_int)
        den = (L / 2.0) - (np.sin(2.0 * k * L) / (4.0 * k))
        A_vals.append(num / den)
    
    return np.array(A_vals)

# Compute eigenvalues and coefficients
L = 10.0
num_modes = 200
k_vals = find_eigenvalues(L, num_modes)
A_vals = compute_coefficients(k_vals, L)

print(f"Computed {len(k_vals)} eigenvalues")

# -----------------------------------------------------------------------------
# Direct 2D spectral evaluation
# -----------------------------------------------------------------------------

def compute_solution_2d_spectral(k_vals, A_vals, t, N_2d=150):
    """
    Compute u(x,y,0,t) directly using spectral series.
    """
    r_max = L
    
    # Create 2D grid
    x = np.linspace(-r_max, r_max, N_2d)
    y = np.linspace(-r_max, r_max, N_2d)
    X, Y = np.meshgrid(x, y)
    
    # Compute radial distance
    R_2d = np.sqrt(X**2 + Y**2)
    
    # Initialize solution
    U_2d = np.zeros_like(R_2d)
    
    # Vectorized evaluation for speed
    for n in range(len(k_vals)):
        k = k_vals[n]
        A = A_vals[n]
        
        # Avoid division by zero at origin
        mask = R_2d > 1e-9
        U_2d[mask] += A * np.cos(k * t) * np.sin(k * R_2d[mask]) / R_2d[mask]
        
        # At origin: lim_{r->0} sin(kr)/r = k
        U_2d[~mask] += A * np.cos(k * t) * k
    
    return X, Y, U_2d

# -----------------------------------------------------------------------------
# Precompute global z-limits
# -----------------------------------------------------------------------------

def compute_global_zlims(k_vals, A_vals, t_1d, N_2d=150):
    """Compute global min/max across all time steps."""
    T_final = len(t_1d) - 1
    time_indices = [0, T_final//3, 2*T_final//3, T_final]
    
    z_min_global = np.inf
    z_max_global = -np.inf
    
    print("Computing global z-limits...")
    for time_idx in time_indices:
        t_val = t_1d[time_idx]
        _, _, U_2d = compute_solution_2d_spectral(k_vals, A_vals, t_val, N_2d)
        z_min_global = min(z_min_global, U_2d.min())
        z_max_global = max(z_max_global, U_2d.max())
    
    print(f"Global z-limits: [{z_min_global:.4f}, {z_max_global:.4f}]")
    return z_min_global, z_max_global

# -----------------------------------------------------------------------------
# Visualization with fixed z-axis
# -----------------------------------------------------------------------------

def plot_3d_evolution_row(k_vals, A_vals, t_1d, N_2d=150):
    """Plot 4 3D surfaces with FIXED z-axis limits and single colorbar."""
    
    # Select time indices
    T_final = len(t_1d) - 1
    time_indices = [0, T_final//3, 2*T_final//3, T_final]
    time_labels = ['0', 'T/3', '2T/3', 'T']
    
    # Compute global z-limits FIRST
    z_min_global, z_max_global = compute_global_zlims(k_vals, A_vals, t_1d, N_2d)
    
    fig = plt.figure(figsize=(12, 3))
    
    surfaces = []  # Store surfaces for shared colorbar
    
    for idx, (time_idx, time_label) in enumerate(zip(time_indices, time_labels), 1):
        t_val = t_1d[time_idx]
        
        print(f"Rendering {time_label}...")
        
        # Compute solution directly using spectral series
        X, Y, U_2d = compute_solution_2d_spectral(k_vals, A_vals, t_val, N_2d)
        
        ax = fig.add_subplot(1, 4, idx, projection='3d')
        
        # Use plot_surface with full resolution
        surf = ax.plot_surface(X, Y, U_2d, 
                               cmap='viridis',
                               edgecolor='none',
                               linewidth=0,
                               antialiased=True,
                               shade=True,
                               rcount=N_2d,
                               ccount=N_2d,
                               vmin=z_min_global,  # Fixed color scale
                               vmax=z_max_global,  # Fixed color scale
                               alpha=1.0)
        
        surfaces.append(surf)
        
        ax.set_title(time_label, fontsize=14)
        
        # Set FIXED limits for all axes
        ax.set_xlim([-L, L])
        ax.set_ylim([-L, L])
        ax.set_zlim([z_min_global, z_max_global])  # FIXED z-axis
        
        ax.view_init(elev=30, azim=45)
        ax.grid(False)
        ax.set_axis_off()
    
    # Add single colorbar on the right
    fig.subplots_adjust(right=0.9)
    cbar_ax = fig.add_axes([0.92, 0.15, 0.02, 0.7])
    fig.colorbar(surfaces[0], cax=cbar_ax)
    
    plt.savefig('spherical_wave_3d_evolution.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\nVisualization saved: spherical_wave_3d_evolution.png")

# -----------------------------------------------------------------------------
# Main Execution
# -----------------------------------------------------------------------------

if __name__ == "__main__":
    print("="*60)
    print("3D Spherical Wave Evolution (Fixed Z-Axis)")
    print("="*60)
    
    import time
    start = time.time()
    plot_3d_evolution_row(k_vals, A_vals, t_1d, N_2d=300)
    print(f"Total time: {time.time() - start:.2f} seconds")
    
    print("="*60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Load the data
data = np.load("../Results/NM_codes/BlackScholes_Dirichlet_CallPayoff_K100/2026_01_11_1202/black_scholes.npz")
t = data['t']       # Time array
x = data['x']       # Space array
u = data['u']       # Price (Time x Space)
gamma = data['gamma'] # Gamma (Time x Space)
theta = data['theta'] # Theta (Time x Space)

# Define target times: 0, T/3, 2T/3, T
T_final = t[-1]
target_times = [0.0, T_final/3.0, 2.0*T_final/3.0, T_final]

# Find indices closest to target times
time_indices = []
for target in target_times:
    idx = (np.abs(t - target)).argmin()
    time_indices.append(idx)

# Setup the plot: 3 rows (Variables) x 4 columns (Time steps)
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
variables = [u, gamma, theta]
var_names = ["Solution (Price)", "Gamma", "Theta"]

for row, (var_data, var_name) in enumerate(zip(variables, var_names)):
    for col, t_idx in enumerate(time_indices):
        ax = axes[row, col]
        
        # Extract profile at specific time index
        # Assuming data shape is (Time, Space) based on Julia transpose
        y_vals = var_data[t_idx, :]
        
        ax.plot(x, y_vals, linewidth=2, color='blue')
        
        # Formatting
        current_time = t[t_idx]
        ax.set_title(f"{var_name}\n$t \\approx {current_time:.2f}$")
        ax.set_xlabel("$x$")
        ax.grid(True, linestyle='--', alpha=0.6)
        
        # Only show y-label on the first column to reduce clutter
        if col == 0:
            ax.set_ylabel(f"${var_name}$")

plt.tight_layout()
plt.savefig("black_scholes_snapshots.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
# We access 'delta' which was saved in the Julia step
data = np.load("../Results/NM_codes/BlackScholes_Dirichlet_CallPayoff_K100/2026_01_11_1202/black_scholes.npz")
t = data['t']
x = data['x']
u = data['u']         # Price
delta = data['delta'] # Delta (New)
gamma = data['gamma'] # Gamma
theta = data['theta'] # Theta

# 2. Define Target Times (0, T/3, 2T/3, T)
T_final = t[-1]
target_times = [0.0, T_final/3.0, 2.0*T_final/3.0, T_final]

# Find nearest indices
indices = []
for target in target_times:
    idx = (np.abs(t - target)).argmin()
    indices.append(idx)

# 3. Plotting: 1 Row x 4 Columns
fig, axes = plt.subplots(1, 4, figsize=(24, 6))

# Define distinct colors for the fields
c_price = 'black'       # u
c_delta = 'red'         # Delta
c_gamma = 'darkorange'  # Gamma
c_theta = 'forestgreen' # Theta

for i, ax in enumerate(axes):
    idx = indices[i]
    current_time = t[idx]
    
    # --- Primary Axis (Left): Price (u) ---
    ln1 = ax.plot(x, u[idx, :], color=c_price, linewidth=2, label=r'$u$ (Price)')
    ax.set_xlabel('x')
    ax.set_title(f"Time $t \\approx {current_time:.2f}$")
    ax.grid(True, alpha=0.3)
    
    # Only label y-axis on the first plot
    if i == 0:
        ax.set_ylabel(r'Price ($u$)', color=c_price)
    
    # --- Secondary Axis (Right): Greeks (Delta, Gamma, Theta) ---
    ax2 = ax.twinx()
    
    ln2 = ax2.plot(x, delta[idx, :], color=c_delta, linewidth=2, linestyle='--', label=r'$\Delta$ (Delta)')
    #ln3 = ax2.plot(x, gamma[idx, :], color=c_gamma, linewidth=2, linestyle=':', label=r'$\Gamma$ (Gamma)')
    ln4 = ax2.plot(x, theta[idx, :], color=c_theta, linewidth=2, linestyle='-.', label=r'$\Theta$ (Theta)')
    
    # Label right axis on the last plot
    if i == 3: 
        ax2.set_ylabel(r'Greeks ($\Delta, \Theta$)', color='black')
    
    # Combine legends
    # We collect handles from both axes to form a single legend box
    lns = ln1 + ln2 + ln4
    labs = [l.get_label() for l in lns]
    ax.legend(lns, labs, loc='center right', fontsize='small', framealpha=0.9)

plt.tight_layout()
plt.savefig("black_scholes_1row_with_delta.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Load Data
data = np.load("../Results/NM_codes/AnisotropicDiffusion_Periodic_SinPlusCos/2026_01_11_0116/anisotropic_diffusion.npz")
t = data['t']
x = data['x']
y = data['y']
usol = data['usol'] # Shape: (Time, X, Y)

# 2. Define Target Times (0, 2/3, 4/3, 2)
T_final = 2.0
target_times = [0.0, T_final/3.0, 2.0*T_final/3.0, T_final]

# Find nearest indices
indices = []
for target in target_times:
    idx = (np.abs(t - target)).argmin()
    indices.append(idx)

# 3. Plotting: 1 Row x 4 Columns
fig, axes = plt.subplots(1, 4, figsize=(24, 5))

# Create meshgrid for plotting (X, Y)
X, Y = np.meshgrid(x, y, indexing='ij') 

for i, ax in enumerate(axes):
    idx = indices[i]
    current_time = t[idx]
    
    # Extract 2D field at this time step
    u_field = usol[idx, :, :]
    
    # Plot Heatmap
    # using pcolormesh for accurate coordinate mapping
    # vmin/vmax fixed to global min/max for consistent color scaling across time
    mesh = ax.pcolormesh(X, Y, u_field, cmap='viridis', shading='auto', 
                         vmin=usol.min(), vmax=usol.max())
    
    ax.set_title(f"Time $t \\approx {current_time:.2f}$")
    ax.set_xlabel('$x$')
    ax.set_aspect('equal')
    
    # Only label y-axis on the first plot
    if i == 0:
        ax.set_ylabel('$y$')
    else:
        ax.set_yticks([])
        
    # Add colorbar for each plot to see value evolution
    fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle(r"2D Anisotropic Diffusion: $u(x,y,t)$", fontsize=16)
plt.tight_layout()
plt.savefig("anisotropic_diffusion_snapshots.png")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Read the NPZ file
data = np.load('../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1605/burgers.npz')
t = data['t']
x = data['x']
usol = data['usol']  # Shape: (Time, Space)

print(f"Data loaded successfully")
print(f"Time array shape: {t.shape}")
print(f"Space array shape: {x.shape}")
print(f"Solution array shape: {usol.shape}")
print(f"Time range: [{t[0]:.3f}, {t[-1]:.3f}]")

# Define the time points to plot
T = t[-1]
target_times = [0.0, T/3, 2*T/3, T]

# Create figure with 1x4 subplots
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for i, (ax, t_target) in enumerate(zip(axes, target_times)):
    # Find the closest time index
    idx = np.argmin(np.abs(t - t_target))
    t_actual = t[idx]
    
    # Plot the solution at this time
    ax.plot(x, usol[idx, :], 'b-', linewidth=2)
    ax.set_title(f't = {t_actual:.3f}', fontsize=12)
    ax.set_xlabel('x', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Only label y-axis on the first subplot
    if i == 0:
        ax.set_ylabel('u', fontsize=11)
    
    # Set consistent y-limits across all plots for better comparison
    ax.set_ylim([usol.min() - 0.1, usol.max() + 0.1])

plt.tight_layout()
plt.savefig('viscous_burgers_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved as 'viscous_burgers_snapshots.png'")

In [ ]:
data = np.load('../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/burgers.npz')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Read the NPZ file
data = np.load('../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/burgers.npz')
t = data['t']
x = data['x']
y = data['y']
usol = data['usol']  # Shape: (Time, X, Y)

print(f"Data loaded successfully")
print(f"Time array shape: {t.shape}")
print(f"X array shape: {x.shape}")
print(f"Y array shape: {y.shape}")
print(f"Solution array shape: {usol.shape}")
print(f"Time snapshots: {t}")

# Create figure with 1x4 subplots for 2D heatmaps
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

# Find global min/max for consistent color scaling
vmin = usol.min()
vmax = usol.max()

for i, (ax, t_val) in enumerate(zip(axes, t)):
    # Extract 2D field at this time snapshot
    u_field = usol[i, :, :]  # Shape: (X, Y)
    
    # Create heatmap
    im = ax.imshow(u_field.T, 
                   extent=[x[0], x[-1], y[0], y[-1]],
                   origin='lower',
                   aspect='equal',
                   cmap='viridis',
                   vmin=vmin,
                   vmax=vmax)
    
    ax.set_title(f't = {t_val:.2f}', fontsize=12)
    
    # Only label axes on the first subplot
    if i == 0:
        ax.set_xlabel('x', fontsize=11)
        ax.set_ylabel('y', fontsize=11)

# Add colorbar only to the rightmost plot
plt.colorbar(im, ax=axes[-1], label='u')

plt.tight_layout()
plt.savefig('burgers_2d_snapshots.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nPlot saved as 'burgers_2d_snapshots.png'")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import simpson
from scipy.optimize import brentq
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ==============================================================================
# VISUAL CONFIGURATION
# ==============================================================================
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 12
mpl.rcParams['figure.dpi'] = 100

# ==============================================================================
# HELPER FUNCTIONS FOR 3D WAVE
# ==============================================================================
def find_eigenvalues(L, num_modes):
    """Find eigenvalues for the spherical wave problem."""
    f = lambda k: np.tan(L*k) - L*k
    k_roots = []
    
    for n in range(1, num_modes + 1):
        z_min = n * np.pi + 1e-6
        z_max = n * np.pi + np.pi/2 - 1e-6
        k_min = z_min / L
        k_max = z_max / L
        
        try:
            root = brentq(f, k_min, k_max)
            k_roots.append(root)
        except:
            pass
    
    return np.array(k_roots)

def compute_coefficients(k_vals, L):
    """Compute Fourier coefficients."""
    r_int = np.linspace(0, L, 10000)
    v0_vals = np.exp(-10.0 * (r_int - 5.0)**2) * np.sin(np.pi * r_int / L)
    
    A_vals = []
    for k in k_vals:
        phi_vals = np.sin(k * r_int)
        integrand = v0_vals * phi_vals
        num = simpson(integrand, x=r_int)
        den = (L / 2.0) - (np.sin(2.0 * k * L) / (4.0 * k))
        A_vals.append(num / den)
    
    return np.array(A_vals)

def compute_solution_2d_spectral(k_vals, A_vals, t, L, N_2d=100):
    """Compute u(x,y,0,t) directly using spectral series."""
    r_max = L
    x = np.linspace(-r_max, r_max, N_2d)
    y = np.linspace(-r_max, r_max, N_2d)
    X, Y = np.meshgrid(x, y)
    R_2d = np.sqrt(X**2 + Y**2)
    U_2d = np.zeros_like(R_2d)
    
    for n in range(len(k_vals)):
        k = k_vals[n]
        A = A_vals[n]
        mask = R_2d > 1e-9
        U_2d[mask] += A * np.cos(k * t) * np.sin(k * R_2d[mask]) / R_2d[mask]
        U_2d[~mask] += A * np.cos(k * t) * k
    
    return X, Y, U_2d

def compute_global_zlims(k_vals, A_vals, t_array, L, time_indices, N_2d=100):
    """Compute global min/max across selected time steps."""
    z_min_global = np.inf
    z_max_global = -np.inf
    
    for time_idx in time_indices:
        t_val = t_array[time_idx]
        _, _, U_2d = compute_solution_2d_spectral(k_vals, A_vals, t_val, L, N_2d)
        z_min_global = min(z_min_global, U_2d.min())
        z_max_global = max(z_max_global, U_2d.max())
    
    return z_min_global, z_max_global

# ==============================================================================
# MAIN PLOTTING FUNCTION
# ==============================================================================

# Create main figure with 5 rows × 4 columns
fig = plt.figure(figsize=(12, 12))

# Column titles (only at the top)
time_labels = ['$0$', r'$T/3$', r'$2T/3$', '$T$']

# ==============================================================================
# ROW 1: BLACK-SCHOLES (1D with Greeks)
# ==============================================================================
print("Loading Black-Scholes data...")
data_bs = np.load("../Results/NM_codes/BlackScholes_Dirichlet_CallPayoff_K100/2026_01_11_1202/black_scholes.npz")
t_bs = data_bs['t']
x_bs = data_bs['x']
u_bs = data_bs['u']
delta_bs = data_bs['delta']
theta_bs = data_bs['theta']

T_bs = t_bs[-1]
target_times_bs = [0.0, T_bs/3, 2*T_bs/3, T_bs]
indices_bs = [np.argmin(np.abs(t_bs - target)) for target in target_times_bs]

c_price = 'black'
c_delta = 'red'
c_theta = 'forestgreen'

for i in range(4):
    ax1 = plt.subplot(5, 4, i+1)
    idx = indices_bs[i]
    
    ln1 = ax1.plot(x_bs, u_bs[idx, :], color=c_price, linewidth=2, label=r'$u$ (Price)')
    ax1.set_title(time_labels[i], fontsize=13)
    #ax1.set_xlabel('$x$', fontsize=13)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(labelsize=12)
    
    if i == 0:
        ax1.set_ylabel('BS1D\n\nPrice ($u$)', color=c_price, fontsize=13)
    
    ax2 = ax1.twinx()
    ln2 = ax2.plot(x_bs, delta_bs[idx, :], color=c_delta, linewidth=2, linestyle='--', label=r'$\Delta$')
    ln4 = ax2.plot(x_bs, theta_bs[idx, :], color=c_theta, linewidth=2, linestyle='-.', label=r'$\Theta$')
    ax2.tick_params(labelsize=12)
    
    if i == 3:
        ax2.set_ylabel(r'Greeks ($\Delta, \Theta$)', color='black', fontsize=13)
    
    lns = ln1 + ln2 + ln4
    labs = [l.get_label() for l in lns]
    if i == 3:
        ax1.legend(lns, labs, loc='center right', fontsize=10, framealpha=0.9)

# ==============================================================================
# ROW 2: VISCOUS BURGERS 1D
# ==============================================================================
print("Loading Viscous Burgers 1D data...")
data_vb = np.load('../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1605/burgers.npz')
t_vb = data_vb['t']
x_vb = data_vb['x']
usol_vb = data_vb['usol']

T_vb = t_vb[-1]
target_times_vb = [0.0, T_vb/3, 2*T_vb/3, T_vb]

for i in range(4):
    ax = plt.subplot(5, 4, 4+i+1)
    idx = np.argmin(np.abs(t_vb - target_times_vb[i]))
    
    ax.plot(x_vb, usol_vb[idx, :], 'b-', linewidth=2)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=12)
    
    if i == 0:
        ax.set_ylabel('VB1D\n\n$u$', fontsize=13)
    
    ax.set_ylim([usol_vb.min() - 0.1, usol_vb.max() + 0.1])

# ==============================================================================
# ROW 3: ANISOTROPIC DIFFUSION 2D
# ==============================================================================
print("Loading Anisotropic Diffusion 2D data...")
data_ad = np.load("../Results/NM_codes/AnisotropicDiffusion_Periodic_SinPlusCos/2026_01_11_0116/anisotropic_diffusion.npz")
t_ad = data_ad['t']
x_ad = data_ad['x']
y_ad = data_ad['y']
usol_ad = data_ad['usol']

T_ad = 2.0
target_times_ad = [0.0, T_ad/3, 2*T_ad/3, T_ad]
indices_ad = [np.argmin(np.abs(t_ad - target)) for target in target_times_ad]

X_ad, Y_ad = np.meshgrid(x_ad, y_ad, indexing='ij')

for i in range(4):
    ax = plt.subplot(5, 4, 8+i+1)
    idx = indices_ad[i]
    u_field = usol_ad[idx, :, :]
    
    mesh = ax.pcolormesh(X_ad, Y_ad, u_field, cmap='jet', shading='auto',
                         vmin=usol_ad.min(), vmax=usol_ad.max())
    
    ax.set_aspect('equal')
    ax.tick_params(labelsize=12)
    
    if i == 0:
        ax.set_ylabel('DE2D\n\n$y$', fontsize=13)
    else:
        ax.set_yticks([])
    
    if i == 3:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(mesh, cax=cax)
        cbar.ax.tick_params(labelsize=12)

# ==============================================================================
# ROW 4: INVISCID BURGERS 2D
# ==============================================================================
print("Loading Inviscid Burgers 2D data...")
data_ib = np.load('../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/burgers.npz')
t_ib = data_ib['t']
x_ib = data_ib['x']
y_ib = data_ib['y']
usol_ib = data_ib['usol']

vmin_ib = usol_ib.min()
vmax_ib = usol_ib.max()

for i in range(4):
    ax = plt.subplot(5, 4, 12+i+1)
    u_field = usol_ib[i, :, :]
    
    im = ax.imshow(u_field.T,
                   extent=[x_ib[0], x_ib[-1], y_ib[0], y_ib[-1]],
                   origin='lower',
                   aspect='equal',
                   cmap='jet',
                   vmin=vmin_ib,
                   vmax=vmax_ib)
    
    ax.tick_params(labelsize=12)
    ax.set_xlabel('$x$', fontsize=13)
    
    if i == 0:
        ax.set_ylabel('IB2D\n\n$y$', fontsize=13)
    else:
        ax.set_yticks([])    
    if i == 3:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=12)

# ==============================================================================
# ROW 5: 3D SPHERICAL WAVE
# ==============================================================================
print("Loading 3D Spherical Wave data...")
data_wave = np.load('../Results/NM_codes/Wave_Neumann_GaussianSin/2026_01_11_1252/wave_spherical.npz')
t_wave = data_wave['t']

# Compute eigenvalues and coefficients
L = 10.0
num_modes = 200
k_vals = find_eigenvalues(L, num_modes)
A_vals = compute_coefficients(k_vals, L)

T_wave = len(t_wave) - 1
time_indices_wave = [0, T_wave//3, 2*T_wave//3, T_wave]

# Compute global z-limits
z_min_global, z_max_global = compute_global_zlims(k_vals, A_vals, t_wave, L, time_indices_wave, N_2d=100)

print("Rendering 3D wave surfaces...")
for idx, time_idx in enumerate(time_indices_wave):
    t_val = t_wave[time_idx]
    
    X, Y, U_2d = compute_solution_2d_spectral(k_vals, A_vals, t_val, L, N_2d=100)
    
    ax = fig.add_subplot(5, 4, 16+idx+1, projection='3d')
    
    surf = ax.plot_surface(X, Y, U_2d,
                          cmap='viridis',
                          edgecolor='none',
                          linewidth=0,
                          antialiased=True,
                          shade=True,
                          rcount=100,
                          ccount=100,
                          vmin=z_min_global,
                          vmax=z_max_global*3/4,
                          alpha=1.0)
    
    ax.set_xlim([-L, L])
    ax.set_ylim([-L, L])
    ax.set_zlim([z_min_global, z_max_global])
    ax.view_init(elev=30, azim=45)
    ax.set_axis_off()
    
    if idx == 0:
        # Add ylabel with positioning
        ax.text2D(-0.1, 0.5, 'WE3D', transform=ax.transAxes, 
                  fontsize=13, va='center', ha='right')
    
    if idx == 3:
        # Use make_axes_locatable for consistent colorbar
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes
        axins = inset_axes(ax, width="5%", height="60%", loc='center right',
                          bbox_to_anchor=(0.15, 0., 1, 1), bbox_transform=ax.transAxes,
                          borderpad=0)
        cbar = plt.colorbar(surf, cax=axins)
        cbar.ax.tick_params(labelsize=12)

# ==============================================================================
# FINAL ADJUSTMENTS AND SAVE
# ==============================================================================
plt.tight_layout()
plt.savefig('combined_all_problems.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCombined visualization saved: combined_all_problems.png")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import simpson
from scipy.optimize import brentq
import matplotlib as mpl
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ==============================================================================
# VISUAL CONFIGURATION
# ==============================================================================
mpl.rcParams['mathtext.fontset'] = 'stix'
mpl.rcParams['font.family'] = 'STIXGeneral'
mpl.rcParams['font.size'] = 12
mpl.rcParams['figure.dpi'] = 100

# ==============================================================================
# HELPER FUNCTIONS FOR 3D WAVE
# ==============================================================================
def find_eigenvalues(L, num_modes):
    """Find eigenvalues for the spherical wave problem."""
    f = lambda k: np.tan(L*k) - L*k
    k_roots = []
    
    for n in range(1, num_modes + 1):
        z_min = n * np.pi + 1e-6
        z_max = n * np.pi + np.pi/2 - 1e-6
        k_min = z_min / L
        k_max = z_max / L
        
        try:
            root = brentq(f, k_min, k_max)
            k_roots.append(root)
        except:
            pass
    
    return np.array(k_roots)

def compute_coefficients(k_vals, L):
    """Compute Fourier coefficients."""
    r_int = np.linspace(0, L, 10000)
    v0_vals = np.exp(-10.0 * (r_int - 5.0)**2) * np.sin(np.pi * r_int / L)
    
    A_vals = []
    for k in k_vals:
        phi_vals = np.sin(k * r_int)
        integrand = v0_vals * phi_vals
        num = simpson(integrand, x=r_int)
        den = (L / 2.0) - (np.sin(2.0 * k * L) / (4.0 * k))
        A_vals.append(num / den)
    
    return np.array(A_vals)

def compute_solution_3d_cut(k_vals, A_vals, t, L, N_3d=80):
    """
    Compute u(x,y,z,t) on a hemisphere with visible interior.
    Shows outer surface AND inner cut plane - properly masked.
    """
    # Create spherical grid for outer surface with higher resolution
    theta_outer = np.linspace(0, np.pi, N_3d)
    phi_outer = np.linspace(0, np.pi, N_3d)  # Only half (0 to pi)
    Theta_outer, Phi_outer = np.meshgrid(theta_outer, phi_outer)
    
    # Outer surface at r = L
    X_outer = L * np.sin(Theta_outer) * np.cos(Phi_outer)
    Y_outer = L * np.sin(Theta_outer) * np.sin(Phi_outer)
    Z_outer = L * np.cos(Theta_outer)
    
    # Compute solution on outer surface
    R_outer = L * np.ones_like(X_outer)
    U_outer = np.zeros_like(X_outer)
    
    for n in range(len(k_vals)):
        k = k_vals[n]
        A = A_vals[n]
        U_outer += A * np.cos(k * t) * np.sin(k * R_outer) / R_outer
    
    # Create cut plane at y = 0 - ONLY inside the sphere
    # Use polar coordinates for a clean circular disk
    r_disk = np.linspace(0, L, N_3d)
    theta_disk = np.linspace(0, 2*np.pi, N_3d)
    R_disk, Theta_disk = np.meshgrid(r_disk, theta_disk)
    
    # Convert to Cartesian for cut plane
    X_cut = R_disk * np.cos(Theta_disk)
    Y_cut = np.zeros_like(X_cut)
    Z_cut = R_disk * np.sin(Theta_disk)
    
    # Compute solution on cut plane
    U_cut = np.zeros_like(X_cut)
    
    for n in range(len(k_vals)):
        k = k_vals[n]
        A = A_vals[n]
        # Avoid division by zero at origin
        mask_nonzero = R_disk > 1e-9
        U_cut[mask_nonzero] += A * np.cos(k * t) * np.sin(k * R_disk[mask_nonzero]) / R_disk[mask_nonzero]
        # Handle origin
        mask_zero = R_disk <= 1e-9
        U_cut[mask_zero] += A * np.cos(k * t) * k
    
    return X_outer, Y_outer, Z_outer, U_outer, X_cut, Y_cut, Z_cut, U_cut

def compute_global_zlims_cut(k_vals, A_vals, t_array, L, time_indices, N_3d=50):
    """Compute global min/max across selected time steps."""
    z_min_global = np.inf
    z_max_global = -np.inf
    
    for time_idx in time_indices:
        t_val = t_array[time_idx]
        _, _, _, U_outer, _, _, _, U_cut = compute_solution_3d_cut(k_vals, A_vals, t_val, L, N_3d)
        z_min_global = min(z_min_global, np.nanmin(U_outer), np.nanmin(U_cut))
        z_max_global = max(z_max_global, np.nanmax(U_outer), np.nanmax(U_cut))
    
    return z_min_global, z_max_global

# ==============================================================================
# MAIN PLOTTING FUNCTION
# ==============================================================================

# Create main figure with 5 rows × 4 columns
fig = plt.figure(figsize=(12, 12))

# Column titles (only at the top)
time_labels = ['$0$', r'$T/3$', r'$2T/3$', '$T$']

# ==============================================================================
# ROW 1: BLACK-SCHOLES (1D with Greeks)
# ==============================================================================
print("Loading Black-Scholes data...")
data_bs = np.load("../Results/NM_codes/BlackScholes_Dirichlet_CallPayoff_K100/2026_01_11_1202/black_scholes.npz")
t_bs = data_bs['t']
x_bs = data_bs['x']
u_bs = data_bs['u']
delta_bs = data_bs['delta']
theta_bs = data_bs['theta']

T_bs = t_bs[-1]
target_times_bs = [0.0, T_bs/3, 2*T_bs/3, T_bs]
indices_bs = [np.argmin(np.abs(t_bs - target)) for target in target_times_bs]

c_price = 'black'
c_delta = 'red'
c_theta = 'forestgreen'

for i in range(4):
    ax1 = plt.subplot(5, 4, i+1)
    idx = indices_bs[i]
    
    ln1 = ax1.plot(x_bs, u_bs[idx, :], color=c_price, linewidth=2, label=r'$u$ (Price)')
    ax1.set_title(time_labels[i], fontsize=13)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(labelsize=12)
    
    if i == 0:
        ax1.set_ylabel('Price ($u$)', color=c_price, fontsize=13)
    
    ax2 = ax1.twinx()
    ln2 = ax2.plot(x_bs, delta_bs[idx, :], color=c_delta, linewidth=2, linestyle='--', label=r'$\Delta$')
    ln4 = ax2.plot(x_bs, theta_bs[idx, :], color=c_theta, linewidth=2, linestyle='-.', label=r'$\Theta$')
    ax2.tick_params(labelsize=12)
    
    if i == 3:
        ax2.set_ylabel(r'Greeks ($\Delta, \Theta$)', color='black', fontsize=13)
    
    lns = ln1 + ln2 + ln4
    labs = [l.get_label() for l in lns]
    if i == 3:
        ax1.legend(lns, labs, loc='center right', fontsize=10, framealpha=0.9)

# ==============================================================================
# ROW 2: VISCOUS BURGERS 1D
# ==============================================================================
print("Loading Viscous Burgers 1D data...")
data_vb = np.load('../Results/NM_codes/Burgers_Periodic_SinPi/2025_12_17_1605/burgers.npz')
t_vb = data_vb['t']
x_vb = data_vb['x']
usol_vb = data_vb['usol']

T_vb = t_vb[-1]
target_times_vb = [0.0, T_vb/3, 2*T_vb/3, T_vb]

for i in range(4):
    ax = plt.subplot(5, 4, 4+i+1)
    idx = np.argmin(np.abs(t_vb - target_times_vb[i]))
    
    ax.plot(x_vb, usol_vb[idx, :], 'b-', linewidth=2)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=12)
    
    if i == 0:
        ax.set_ylabel('$u$', fontsize=13)
    
    ax.set_ylim([usol_vb.min() - 0.1, usol_vb.max() + 0.1])

# ==============================================================================
# ROW 3: ANISOTROPIC DIFFUSION 2D
# ==============================================================================
print("Loading Anisotropic Diffusion 2D data...")
data_ad = np.load("../Results/NM_codes/AnisotropicDiffusion_Periodic_SinPlusCos/2026_01_11_0116/anisotropic_diffusion.npz")
t_ad = data_ad['t']
x_ad = data_ad['x']
y_ad = data_ad['y']
usol_ad = data_ad['usol']

T_ad = 2.0
target_times_ad = [0.0, T_ad/3, 2*T_ad/3, T_ad]
indices_ad = [np.argmin(np.abs(t_ad - target)) for target in target_times_ad]

X_ad, Y_ad = np.meshgrid(x_ad, y_ad, indexing='ij')

for i in range(4):
    ax = plt.subplot(5, 4, 8+i+1)
    idx = indices_ad[i]
    u_field = usol_ad[idx, :, :]
    
    mesh = ax.pcolormesh(X_ad, Y_ad, u_field, cmap='jet', shading='auto',
                         vmin=usol_ad.min(), vmax=usol_ad.max())
    
    ax.set_aspect('equal')
    ax.tick_params(labelsize=12)
    
    if i == 0:
        ax.set_ylabel('$y$', fontsize=13)
    else:
        ax.set_yticks([])
    
    if i == 3:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(mesh, cax=cax)
        cbar.ax.tick_params(labelsize=12)

# ==============================================================================
# ROW 4: INVISCID BURGERS 2D
# ==============================================================================
print("Loading Inviscid Burgers 2D data...")
data_ib = np.load('../Results/NM_codes/Burgers_Periodic_SinSumHalf/2025_12_23_1233/burgers.npz')
t_ib = data_ib['t']
x_ib = data_ib['x']
y_ib = data_ib['y']
usol_ib = data_ib['usol']

vmin_ib = usol_ib.min()
vmax_ib = usol_ib.max()

for i in range(4):
    ax = plt.subplot(5, 4, 12+i+1)
    u_field = usol_ib[i, :, :]
    
    im = ax.imshow(u_field.T,
                   extent=[x_ib[0], x_ib[-1], y_ib[0], y_ib[-1]],
                   origin='lower',
                   aspect='equal',
                   cmap='jet',
                   vmin=vmin_ib,
                   vmax=vmax_ib)
    
    ax.tick_params(labelsize=12)
    ax.set_xlabel('$x$', fontsize=13)
    
    if i == 0:
        ax.set_ylabel('$y$', fontsize=13)
    else:
        ax.set_yticks([])    
    if i == 3:
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=12)

# ==============================================================================
# ROW 5: 3D SPHERICAL WAVE WITH CUT PLANE
# ==============================================================================
print("Loading 3D Spherical Wave data...")
data_wave = np.load('../Results/NM_codes/Wave_Neumann_GaussianSin/2026_01_11_1252/wave_spherical.npz')
t_wave = data_wave['t']

# Compute eigenvalues and coefficients
L = 10.0
num_modes = 200
k_vals = find_eigenvalues(L, num_modes)
A_vals = compute_coefficients(k_vals, L)

T_wave = len(t_wave) - 1
time_indices_wave = [0, T_wave//3, 2*T_wave//3, T_wave]

# Compute global z-limits with HIGHER RESOLUTION
N_3d = 120  # Increased from 60
z_min_global, z_max_global = compute_global_zlims_cut(k_vals, A_vals, t_wave, L, time_indices_wave, N_3d)

print(f"Global value range: [{z_min_global:.6f}, {z_max_global:.6f}]")
print("Rendering 3D wave with hemisphere cut and interior plane...")

for idx, time_idx in enumerate(time_indices_wave):
    t_val = t_wave[time_idx]
    
    X_outer, Y_outer, Z_outer, U_outer, X_cut, Y_cut, Z_cut, U_cut = compute_solution_3d_cut(k_vals, A_vals, t_val, L, N_3d)
    
    ax = fig.add_subplot(5, 4, 16+idx+1, projection='3d')
    
    # Normalize colors
    norm = plt.Normalize(vmin=z_min_global, vmax=z_max_global*3/4)
    
    # Plot hemisphere outer surface (semi-transparent) with SMOOTHER rendering
    colors_outer = plt.cm.jet(norm(U_outer))
    surf1 = ax.plot_surface(X_outer, Y_outer, Z_outer, facecolors=colors_outer,
                           shade=True, alpha=1.0, antialiased=True,
                           rcount=N_3d, ccount=N_3d, linewidth=0)
    
    # Plot cut plane interior (opaque) with SMOOTHER rendering
    colors_cut = plt.cm.jet(norm(U_cut))
    surf2 = ax.plot_surface(X_cut, Y_cut, Z_cut, facecolors=colors_cut,
                           shade=True, alpha=1.0, antialiased=True,
                           rcount=N_3d, ccount=N_3d, linewidth=0)
    
    # Set limits and view
    ax.set_xlim([-L, L])
    ax.set_ylim([-L, L])
    ax.set_zlim([-L, L])
    ax.view_init(elev=25, azim=-70)
    ax.set_axis_off()
    
    # if idx == 0:
    #     ax.text2D(-0.1, 0.5, 'WE3D', transform=ax.transAxes, 
    #               fontsize=13, va='center', ha='right')
    
    if idx == 3:
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes
        axins = inset_axes(ax, width="5%", height="60%", loc='center right',
                          bbox_to_anchor=(0.15, 0., 1, 1), bbox_transform=ax.transAxes,
                          borderpad=0)
        import matplotlib.cm as cm
        sm = cm.ScalarMappable(cmap='jet', norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, cax=axins)
        cbar.ax.tick_params(labelsize=12)

# ==============================================================================
# FINAL ADJUSTMENTS AND SAVE
# ==============================================================================
plt.tight_layout()
plt.savefig('combined_all_problems.png', dpi=200, bbox_inches='tight')
plt.show()

print("\nCombined visualization saved: combined_all_problems.png")